# Financial Health Index (FHI) Prediction
### Zindi Competition — Southern African SME Financial Health Classification

---

## Competition Overview

Across Southern Africa, small and medium-sized enterprises (SMEs) are vital to employment, innovation, and economic growth — yet many remain financially fragile and excluded from formal financial systems. This competition introduces a **Financial Health Index (FHI)**: a composite measure classifying businesses into **Low**, **Medium**, or **High** financial health across four dimensions:

- Savings and assets
- Debt and repayment ability
- Resilience to shocks
- Access to credit and financial services

| | |
|---|---|
| **Task** | Multiclass classification (Low / Medium / High) |
| **Metric** | Macro F1 Score |
| **Countries** | Eswatini, Lesotho, Zimbabwe, Malawi |
| **Public LB** | 0.8847 |
| **Private LB** | **0.8860** ✅ |

---

## Solution Pipeline

```
Raw Survey Data
      │
      ▼
 Data Cleaning          ← apostrophe normalization, encoding fixes, placeholder values
      │
      ▼
 Feature Engineering    ← log transforms, financial ratios, missing flags, composite scores
      │
      ▼
 SMOTE (per fold)       ← oversample minority High class inside each CV fold
      │
      ▼
 5-Fold OOF Training    ← XGBoost + LightGBM + CatBoost, each Optuna-tuned (50 trials)
      │
      ▼
 Weighted Ensemble      ← weights proportional to OOF F1 score
      │
      ▼
 Threshold Optimization ← Nelder-Mead on OOF predictions to maximize Macro F1
      │
      ▼
 Submission
```

---
## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
SEED        = 42
N_FOLDS     = 5
N_TRIALS    = 50
TARGET_COL  = "Target"
ID_COL      = "ID"
LABEL_ORDER = ["Low", "Medium", "High"]

# Update DATA_DIR to point to your local data files
DATA_DIR   = Path("/kaggle/input/datasets/frankmandele/financial-health-prediction")
OUTPUT_DIR = Path("/kaggle/working")

# Label encoder sorts alphabetically: High=0, Low=1, Medium=2
le_target = LabelEncoder()
le_target.fit(LABEL_ORDER)

print("=" * 60)
print("FINANCIAL HEALTH INDEX PREDICTION — WINNING SOLUTION")
print("=" * 60)
print(f"Target classes : {le_target.classes_}")
print(f"N_FOLDS        : {N_FOLDS}")
print(f"N_TRIALS       : {N_TRIALS}")

FINANCIAL HEALTH INDEX PREDICTION — WINNING SOLUTION
Target classes : ['High' 'Low' 'Medium']
N_FOLDS        : 5
N_TRIALS       : 50


---
## 2. Load Data

In [ ]:
print("\n1. Loading data...")
train = pd.read_csv(DATA_DIR / "Train.csv")
test  = pd.read_csv(DATA_DIR / "Test.csv")

print(f"   Train : {train.shape}")
print(f"   Test  : {test.shape}")
print(f"   Target distribution:\n{train[TARGET_COL].value_counts().to_string()}")


1. Loading data...
   Train : (9618, 39)
   Test  : (2405, 38)
   Target distribution:
Target
Low       6280
Medium    2868
High       470


---
## 3. Data Cleaning & Preprocessing

A thorough data audit revealed several encoding issues that were silently corrupting features before any modeling. The fixes below were the single most impactful change in the entire competition.

| Fix | Issue | Impact |
|-----|-------|--------|
| Apostrophe normalization | Curly `'` (Unicode 8217) vs straight `'` (Unicode 39) caused silent `NaN` mapping across 9 status columns | Unlocked true signal in `medical_insurance`, `funeral_insurance`, and 7 others |
| Don't know unification | Multiple inconsistent variants (`"Don't Know"`, `"Don?t know"`, `"Refused"`, etc.) | Consistent `-1` encoding across all columns |
| `current_problem_cash_flow` | String `"0"` used alongside `"Yes"`/`"No"` (data entry artifact) | Mapped `"0"` → `"No"` |
| `owner_age` placeholders | Values 99 and 103 are survey refusal codes, not real ages | Replaced with `NaN` then median-imputed |
| `keeps_financial_records` | Contained `"Yes, always"` and `"Yes, sometimes"` as separate values | Unified to `"Yes"` |

In [ ]:
def clean_and_preprocess(df):
    """
    Clean and encode raw survey data.

    Applies six targeted fixes identified during data audit, then encodes
    all categorical columns into numeric representations suitable for
    gradient boosting models.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe with Target and ID columns already removed.

    Returns
    -------
    pd.DataFrame
        Cleaned and encoded dataframe ready for feature engineering.
    """
    df = df.copy()

    # -------------------------------------------------------------------------
    # Fix 1: Normalize apostrophe variants (curly Unicode 8217 → straight 39)
    # This was silently causing NaN in 9 status columns during .map() calls
    # -------------------------------------------------------------------------
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.replace('\u2019', "'", regex=False)

    # -------------------------------------------------------------------------
    # Fix 2: Strip leading/trailing whitespace
    # Prevents invisible whitespace from causing encoding mismatches
    # -------------------------------------------------------------------------
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    # -------------------------------------------------------------------------
    # Fix 3: Unify all "don't know" variants to a single "Unknown" value
    # Multiple representations existed across columns and countries
    # -------------------------------------------------------------------------
    dont_know_variants = [
        "Don't know", "Don't Know", "Don't know or N/A",
        "Don't know (Do not show)", "Don?t know / doesn?t apply",
        "Do not know / N\u200e/A", " Do not know / N\u200e/A",
        "Refused"
    ]
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].replace(dont_know_variants, "Unknown")

    # -------------------------------------------------------------------------
    # Fix 4: current_problem_cash_flow — "0" is a data entry artifact for "No"
    # -------------------------------------------------------------------------
    if "current_problem_cash_flow" in df.columns:
        df["current_problem_cash_flow"] = df["current_problem_cash_flow"].replace({"0": "No"})

    # -------------------------------------------------------------------------
    # Fix 5: owner_age — 99 and 103 are survey refusal placeholders, not ages
    # -------------------------------------------------------------------------
    if "owner_age" in df.columns:
        df["owner_age"] = df["owner_age"].apply(
            lambda x: np.nan if pd.notna(x) and x >= 99 else x
        )

    # -------------------------------------------------------------------------
    # Fix 6: keeps_financial_records — granular values unified to binary Yes/No
    # -------------------------------------------------------------------------
    if "keeps_financial_records" in df.columns:
        df["keeps_financial_records"] = df["keeps_financial_records"].replace({
            "Yes, always": "Yes",
            "Yes, sometimes": "Yes"
        })

    # -------------------------------------------------------------------------
    # Encode: Status columns → ordinal integers
    # Natural ordering: Have now (2) > Used to have (1) > Never had (0)
    # "Unknown" responses encoded as -1 to distinguish from NaN (missing)
    # -------------------------------------------------------------------------
    status_map = {
        "Have now"                       : 2,
        "Used to have but don't have now": 1,
        "Never had"                      : 0,
        "Unknown"                        : -1,
    }
    status_cols = [
        "motor_vehicle_insurance", "has_mobile_money", "has_credit_card",
        "has_loan_account", "has_internet_banking", "has_debit_card",
        "medical_insurance", "funeral_insurance",
        "uses_friends_family_savings", "uses_informal_lender"
    ]
    for col in status_cols:
        if col in df.columns:
            df[col] = df[col].map(status_map)  # unmapped values → NaN automatically

    # -------------------------------------------------------------------------
    # Encode: Binary Yes/No columns → 1 / 0 / -1
    # -------------------------------------------------------------------------
    binary_map = {"Yes": 1, "No": 0, "Unknown": -1}
    binary_cols = [
        "attitude_stable_business_environment", "attitude_worried_shutdown",
        "compliance_income_tax", "perception_insurance_doesnt_cover_losses",
        "perception_cannot_afford_insurance", "has_cellphone",
        "attitude_satisfied_with_achievement", "keeps_financial_records",
        "perception_insurance_companies_dont_insure_businesses_like_yours",
        "perception_insurance_important", "has_insurance",
        "covid_essential_service", "attitude_more_successful_next_year",
        "problem_sourcing_money", "marketing_word_of_mouth",
        "future_risk_theft_stock", "motivation_make_more_money",
        "current_problem_cash_flow"
    ]
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].map(binary_map)

    # -------------------------------------------------------------------------
    # Encode: owner_sex → binary (Male=1, Female=0)
    # -------------------------------------------------------------------------
    if "owner_sex" in df.columns:
        df["owner_sex"] = df["owner_sex"].map({"Male": 1, "Female": 0})

    # -------------------------------------------------------------------------
    # Encode: offers_credit_to_customers → ordinal (Always=2, Sometimes=1, No=0)
    # -------------------------------------------------------------------------
    if "offers_credit_to_customers" in df.columns:
        df["offers_credit_to_customers"] = df["offers_credit_to_customers"].map({
            "Yes, always": 2, "Yes, sometimes": 1, "No": 0
        })

    # -------------------------------------------------------------------------
    # Encode: country → one-hot
    # Country is highly predictive (Eswatini 11.5% High vs Lesotho 0.3%)
    # -------------------------------------------------------------------------
    if "country" in df.columns:
        df = pd.get_dummies(df, columns=["country"], prefix="country", drop_first=False)

    return df


# Apply preprocessing to train and test
train_fe = clean_and_preprocess(train.drop(columns=[TARGET_COL, ID_COL]))
test_fe  = clean_and_preprocess(test.drop(columns=[ID_COL]))

# Align columns — handles any one-hot encoding discrepancies between splits
train_fe, test_fe = train_fe.align(test_fe, join="left", axis=1, fill_value=0)

# Encode target labels
y = le_target.transform(train[TARGET_COL])

# Impute remaining NaNs with median — fit on train only to prevent leakage
imputer = SimpleImputer(strategy="median")
X       = imputer.fit_transform(train_fe.values)
X_test  = imputer.transform(test_fe.values)

feature_names = list(train_fe.columns)

print(f"Features: {len(feature_names)}")
print(f"\nNaN in X: {np.isnan(X).sum()}")
print(f"NaN in X_test: {np.isnan(X_test).sum()}")

Features: 40

NaN in X: 0
NaN in X_test: 0


---
## 4. Feature Engineering

Additional features constructed on top of the preprocessed data to capture patterns the raw survey fields cannot express directly.

| Feature Group | Features | Rationale |
|---|---|---|
| Log-transformed financials | `log_personal_income`, `log_business_expenses`, `log_business_turnover` | Multi-currency data spans 6 orders of magnitude — log scale normalizes |
| Financial ratios | `profit_proxy`, `expense_ratio`, `income_to_turnover`, `income_to_expenses` | Relative financial health is more meaningful than absolute currency values |
| Business age | `total_business_months` | Combines years + months into one continuous feature |
| Missing flags | `missing_*` (×15) | Missingness pattern is informative — Lesotho respondents have systematic gaps correlated with Low FHI |
| Financial product count | `n_financial_products` | Active product ownership signals integration into formal finance |
| Attitude score | `attitude_score` | Optimistic owners tend to run more financially healthy businesses |

In [ ]:
def engineer_features(df):
    """
    Add engineered features on top of preprocessed data.

    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed dataframe from clean_and_preprocess().

    Returns
    -------
    pd.DataFrame
        Dataframe with additional engineered features appended.
    """
    df = df.copy()

    # -------------------------------------------------------------------------
    # Log-transform skewed financial columns
    # Income spans 0 to 150M with a median of 2K — log scale is essential
    # -------------------------------------------------------------------------
    for col in ["personal_income", "business_expenses", "business_turnover"]:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(df[col].fillna(0))

    # -------------------------------------------------------------------------
    # Financial ratio features
    # Ratios are currency-invariant — critical for multi-country data
    # -------------------------------------------------------------------------
    eps = 1e-6  # small constant to prevent division by zero
    df["profit_proxy"]       = df["business_turnover"].fillna(0) - df["business_expenses"].fillna(0)
    df["log_profit_proxy"]   = np.log1p(np.maximum(df["profit_proxy"], 0))
    df["expense_ratio"]      = df["business_expenses"].fillna(0) / (df["business_turnover"].fillna(0) + eps)
    df["income_to_turnover"] = df["personal_income"].fillna(0)   / (df["business_turnover"].fillna(0) + eps)
    df["income_to_expenses"] = df["personal_income"].fillna(0)   / (df["business_expenses"].fillna(0) + eps)

    # -------------------------------------------------------------------------
    # Business age in total months
    # Merges business_age_years and business_age_months into one feature
    # -------------------------------------------------------------------------
    df["total_business_months"] = (
        df["business_age_years"].fillna(0) * 12 +
        df["business_age_months"].fillna(0)
    )

    # -------------------------------------------------------------------------
    # Missing value flags for high-missingness columns
    # WHO didn't answer is often as predictive as what they answered
    # missing_business_age_years was the 2nd most important SHAP feature
    # -------------------------------------------------------------------------
    high_missing = [
        "motor_vehicle_insurance", "has_mobile_money", "current_problem_cash_flow",
        "has_cellphone", "has_loan_account", "has_internet_banking", "has_debit_card",
        "future_risk_theft_stock", "medical_insurance", "funeral_insurance",
        "motivation_make_more_money", "uses_friends_family_savings",
        "uses_informal_lender", "business_age_months", "business_age_years"
    ]
    for col in high_missing:
        if col in df.columns:
            df[f"missing_{col}"] = df[col].isna().astype(int)

    # -------------------------------------------------------------------------
    # Count of financial products currently owned (value == 2 means "Have now")
    # Businesses with more active products signal stronger formal finance access
    # -------------------------------------------------------------------------
    product_cols = [
        "motor_vehicle_insurance", "has_mobile_money", "has_credit_card",
        "has_loan_account", "has_internet_banking", "has_debit_card",
        "medical_insurance", "funeral_insurance"
    ]
    present = [c for c in product_cols if c in df.columns]
    df["n_financial_products"] = df[present].apply(
        lambda row: (row == 2).sum(), axis=1
    )

    # -------------------------------------------------------------------------
    # Positive attitude score (0–3)
    # Counts optimistic survey responses — correlates with higher FHI
    # -------------------------------------------------------------------------
    attitude_cols = [
        "attitude_stable_business_environment",
        "attitude_more_successful_next_year",
        "attitude_satisfied_with_achievement"
    ]
    present_att = [c for c in attitude_cols if c in df.columns]
    df["attitude_score"] = df[present_att].apply(
        lambda row: (row == 1).sum(), axis=1
    )

    return df


# Apply feature engineering on top of preprocessed data
train_fe = clean_and_preprocess(train.drop(columns=[TARGET_COL, ID_COL]))
train_fe = engineer_features(train_fe)

test_fe  = clean_and_preprocess(test.drop(columns=[ID_COL]))
test_fe  = engineer_features(test_fe)

# Align columns after one-hot encoding
train_fe, test_fe = train_fe.align(test_fe, join="left", axis=1, fill_value=0)

# Re-encode target and re-impute on the expanded feature set
y = le_target.transform(train[TARGET_COL])

imputer = SimpleImputer(strategy="median")
X       = imputer.fit_transform(train_fe.values)
X_test  = imputer.transform(test_fe.values)

feature_names = list(train_fe.columns)

print(f"Features after engineering: {len(feature_names)}")
print(f"NaN in X: {np.isnan(X).sum()}")
print(f"NaN in X_test: {np.isnan(X_test).sum()}")

Features after engineering: 66
NaN in X: 0
NaN in X_test: 0


---
## 5. Class Imbalance — SMOTE

The target variable is severely imbalanced:

| Class | Count | Share |
|-------|-------|-------|
| Low | 6,280 | 65.3% |
| Medium | 2,868 | 29.8% |
| **High** | **470** | **4.9%** |

Without correction, models learn to predict "Low" almost exclusively and achieve poor macro F1. We use **SMOTE** with `strategy="minority"` — oversampling only the High class up to match the Medium class count. Critically, SMOTE is applied **inside each CV fold on training data only** so validation folds always evaluate on original unaugmented data.

In [ ]:
# SMOTE applied per fold inside each training loop
# strategy="minority" oversamples only the smallest class (High)
sm = SMOTE(sampling_strategy="minority", random_state=SEED, k_neighbors=5)
print("SMOTE initialized — strategy: minority (oversample High class to match Medium)")

SMOTE initialized — strategy: minority (oversample High class to match Medium)


---
## 6. Cross-Validation & OOF Training Utilities

We use **5-fold stratified cross-validation** with out-of-fold (OOF) predictions. Each model produces:
- `oof_probs` — predictions on held-out validation data (shape: `n_train × 3`)
- `test_probs` — averaged predictions across all folds on test data (shape: `n_test × 3`)

OOF predictions serve two purposes:
1. **Honest model evaluation** — reflects true generalization without test set leakage
2. **Ensemble weighting** — each model's weight in the final ensemble is proportional to its OOF F1

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def macro_f1(y_true, y_pred):
    """Compute macro-averaged F1 score."""
    return f1_score(y_true, y_pred, average="macro")


def train_oof(model_cls, params, X, y, X_test, sm=None, label="Model"):
    """
    Generic OOF training loop for XGBoost-style models.

    Trains a model on each fold's training data (with optional SMOTE),
    evaluates on held-out validation data, and averages test predictions
    across folds.

    Parameters
    ----------
    model_cls  : sklearn-compatible classifier class
    params     : dict of model hyperparameters
    X, y       : training features and labels
    X_test     : test features
    sm         : SMOTE instance (applied per fold if provided)
    label      : string label for print output

    Returns
    -------
    oof, test_preds, oof_score : OOF probabilities, test probabilities, macro F1
    """
    oof   = np.zeros((len(y), 3))
    t_out = np.zeros((len(X_test), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]

        # Apply SMOTE on training fold only — validation uses original data
        if sm is not None:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        # Recompute sample weights after SMOTE (class distribution has changed)
        sw = compute_sample_weight("balanced", y_tr)

        m = model_cls(**params, early_stopping_rounds=50)
        m.fit(X_tr, y_tr,
              sample_weight=sw,
              eval_set=[(X[val_idx], y[val_idx])],
              verbose=False)

        oof[val_idx] = m.predict_proba(X[val_idx])
        t_out       += m.predict_proba(X_test) / N_FOLDS
        print(f"  [{label}] Fold {fold+1} F1: {macro_f1(y[val_idx], np.argmax(oof[val_idx], axis=1)):.4f}")

    score = macro_f1(y, np.argmax(oof, axis=1))
    print(f"\n  [{label}] OOF Macro F1: {score:.4f}")
    print(classification_report(y, np.argmax(oof, axis=1), target_names=le_target.classes_))
    return oof, t_out, score


print("CV utility ready.")

CV utility ready.


---
## 7. XGBoost — Optuna Tuning + OOF Training

**Optuna** runs 50 trials of Bayesian hyperparameter search using the TPE sampler. Each trial trains a 5-fold CV model and returns the mean macro F1 as the objective. The best parameters are then used to train the final OOF model.

> **Reproducibility note:** XGBoost's `hist` tree method uses parallel floating-point operations whose ordering differs between GPU and CPU, producing slightly different results across hardware. GPU execution yields higher scores on this dataset.

In [ ]:
def xgb_objective(trial):
    """Optuna objective: returns mean 5-fold macro F1 for a given XGBoost config."""
    params = {
        "objective"        : "multi:softprob",
        "num_class"        : 3,
        "eval_metric"      : "mlogloss",
        "verbosity"        : 0,
        "use_label_encoder": False,
        "random_state"     : SEED,
        "tree_method"      : "hist",
        "n_estimators"     : trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth"        : trial.suggest_int("max_depth", 3, 10),
        "min_child_weight" : trial.suggest_int("min_child_weight", 1, 20),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma"            : trial.suggest_float("gamma", 1e-4, 5.0, log=True),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = sm.fit_resample(X[tr_idx], y[tr_idx])
        sw = compute_sample_weight("balanced", y_tr)
        m  = xgb.XGBClassifier(**params, early_stopping_rounds=50)
        m.fit(X_tr, y_tr, sample_weight=sw,
              eval_set=[(X[val_idx], y[val_idx])], verbose=False)
        scores.append(macro_f1(y[val_idx], m.predict(X[val_idx])))
    return np.mean(scores)


print(f"Running Optuna hyperparameter search ({N_TRIALS} trials)...")
study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest Optuna F1 : {study_xgb.best_value:.4f}")
print(f"Best params    : {study_xgb.best_params}")

Running Optuna hyperparameter search (50 trials)...

Best Optuna F1 : 0.8046
Best params    : {'n_estimators': 302, 'learning_rate': 0.0888, 'max_depth': 5, 'min_child_weight': 1, 'subsample': 0.784, 'colsample_bytree': 0.671, 'gamma': 0.172, 'reg_alpha': 0.301, 'reg_lambda': 0.001}


In [ ]:
# Add fixed parameters back to the Optuna best params dictionary
best_xgb_params = study_xgb.best_params
best_xgb_params.update({
    "objective"        : "multi:softprob",
    "num_class"        : 3,
    "eval_metric"      : "mlogloss",
    "verbosity"        : 0,
    "use_label_encoder": False,
    "random_state"     : SEED,
    "tree_method"      : "hist",
})

print("Training XGBoost with best Optuna params + SMOTE...")
oof_xgb, test_xgb, xgb_score = train_oof(
    xgb.XGBClassifier, best_xgb_params, X, y, X_test,
    sm=sm, label="XGB"
)

Training XGBoost with best Optuna params + SMOTE...
  [XGB] Fold 1 F1: 0.8026
  [XGB] Fold 2 F1: 0.8328
  [XGB] Fold 3 F1: 0.7885
  [XGB] Fold 4 F1: 0.7971
  [XGB] Fold 5 F1: 0.8021

  [XGB] OOF Macro F1: 0.8050
              precision    recall  f1-score   support

        High       0.88      0.61      0.72       470
         Low       0.92      0.90      0.91      6280
      Medium       0.75      0.82      0.78      2868

    accuracy                           0.86      9618
   macro avg       0.85      0.78      0.80      9618
weighted avg       0.87      0.86      0.86      9618



---
## 8. LightGBM — Optuna Tuning + OOF Training

LightGBM uses leaf-wise tree growth which captures complex non-linear patterns more efficiently than XGBoost's depth-wise growth. The `class_weight="balanced"` parameter provides additional imbalance correction alongside SMOTE. Note that LightGBM uses callbacks for early stopping rather than the `early_stopping_rounds` constructor argument used by XGBoost.

In [ ]:
def lgb_objective(trial):
    """Optuna objective: returns mean 5-fold macro F1 for a given LightGBM config."""
    params = {
        "objective"        : "multiclass",
        "num_class"        : 3,
        "metric"           : "multi_logloss",
        "verbosity"        : -1,
        "boosting_type"    : "gbdt",
        "random_state"     : SEED,
        "class_weight"     : "balanced",
        "n_estimators"     : trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves"       : trial.suggest_int("num_leaves", 20, 150),
        "max_depth"        : trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = sm.fit_resample(X[tr_idx], y[tr_idx])
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=[(X[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
        scores.append(macro_f1(y[val_idx], m.predict(X[val_idx])))
    return np.mean(scores)


print(f"Running Optuna for LightGBM ({N_TRIALS} trials)...")
study_lgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest LGB Optuna F1 : {study_lgb.best_value:.4f}")
print(f"Best params        : {study_lgb.best_params}")

Running Optuna for LightGBM (50 trials)...

Best LGB Optuna F1 : 0.8051
Best params        : {'n_estimators': 774, 'learning_rate': 0.0323, 'num_leaves': 140, 'max_depth': 6, 'min_child_samples': 65, 'subsample': 0.701, 'colsample_bytree': 0.503, 'reg_alpha': 0.001, 'reg_lambda': 0.001}


In [ ]:
def train_oof_lgb(params, X, y, X_test, sm=None, label="LGB"):
    """
    OOF training loop for LightGBM.

    Separate from train_oof() because LightGBM uses callbacks for early
    stopping rather than the early_stopping_rounds constructor argument.
    """
    oof   = np.zeros((len(y), 3))
    t_out = np.zeros((len(X_test), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        if sm is not None:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=[(X[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])

        oof[val_idx] = m.predict_proba(X[val_idx])
        t_out       += m.predict_proba(X_test) / N_FOLDS
        print(f"  [{label}] Fold {fold+1} F1: {macro_f1(y[val_idx], np.argmax(oof[val_idx], axis=1)):.4f}")

    score = macro_f1(y, np.argmax(oof, axis=1))
    print(f"\n  [{label}] OOF Macro F1: {score:.4f}")
    print(classification_report(y, np.argmax(oof, axis=1), target_names=le_target.classes_))
    return oof, t_out, score


best_lgb_params = study_lgb.best_params
best_lgb_params.update({
    "objective"    : "multiclass", "num_class"   : 3,
    "metric"       : "multi_logloss", "verbosity": -1,
    "boosting_type": "gbdt", "random_state"       : SEED,
    "class_weight" : "balanced"
})

print("Training LightGBM with best Optuna params + SMOTE...")
oof_lgb, test_lgb, lgb_score = train_oof_lgb(
    best_lgb_params, X, y, X_test, sm=sm, label="LGB"
)

Training LightGBM with best Optuna params + SMOTE...
  [LGB] Fold 1 F1: 0.8022
  [LGB] Fold 2 F1: 0.8278
  [LGB] Fold 3 F1: 0.7897
  [LGB] Fold 4 F1: 0.8048
  [LGB] Fold 5 F1: 0.8010

  [LGB] OOF Macro F1: 0.8054
              precision    recall  f1-score   support

        High       0.90      0.60      0.72       470
         Low       0.93      0.90      0.91      6280
      Medium       0.75      0.84      0.79      2868

    accuracy                           0.86      9618
   macro avg       0.86      0.78      0.81      9618
weighted avg       0.87      0.86      0.87      9618



---
## 9. CatBoost — Optuna Tuning + OOF Training

CatBoost uses an ordered boosting approach that reduces prediction shift and overfitting. Setting `eval_metric="TotalF1"` allows CatBoost to optimize directly for the competition metric during training rather than log-loss.

In [ ]:
def cat_objective(trial):
    """Optuna objective: returns mean 5-fold macro F1 for a given CatBoost config."""
    params = {
        "loss_function"      : "MultiClass",
        "eval_metric"        : "TotalF1",   # optimize directly for the competition metric
        "random_seed"        : SEED,
        "verbose"            : 0,
        "auto_class_weights" : "Balanced",
        "iterations"         : trial.suggest_int("iterations", 200, 1000),
        "learning_rate"      : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth"              : trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg"        : trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength"    : trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = sm.fit_resample(X[tr_idx], y[tr_idx])
        m = CatBoostClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=(X[val_idx], y[val_idx]),
              early_stopping_rounds=50)
        scores.append(macro_f1(y[val_idx], m.predict(X[val_idx]).flatten()))
    return np.mean(scores)


print(f"Running Optuna for CatBoost ({N_TRIALS} trials)...")
study_cat = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study_cat.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest CAT Optuna F1 : {study_cat.best_value:.4f}")
print(f"Best params        : {study_cat.best_params}")

Running Optuna for CatBoost (50 trials)...

Best CAT Optuna F1 : 0.7987
Best params        : {'iterations': 957, 'learning_rate': 0.0400, 'depth': 10, 'l2_leaf_reg': 0.001, 'bagging_temperature': 0.598, 'random_strength': 0.577}


In [ ]:
def train_oof_cat(params, X, y, X_test, sm=None, label="CAT"):
    """OOF training loop for CatBoost."""
    oof   = np.zeros((len(y), 3))
    t_out = np.zeros((len(X_test), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        if sm is not None:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        m = CatBoostClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=(X[val_idx], y[val_idx]),
              early_stopping_rounds=50)

        oof[val_idx] = m.predict_proba(X[val_idx])
        t_out       += m.predict_proba(X_test) / N_FOLDS
        print(f"  [{label}] Fold {fold+1} F1: {macro_f1(y[val_idx], np.argmax(oof[val_idx], axis=1)):.4f}")

    score = macro_f1(y, np.argmax(oof, axis=1))
    print(f"\n  [{label}] OOF Macro F1: {score:.4f}")
    print(classification_report(y, np.argmax(oof, axis=1), target_names=le_target.classes_))
    return oof, t_out, score


best_cat_params = study_cat.best_params
best_cat_params.update({
    "loss_function"     : "MultiClass",
    "eval_metric"       : "TotalF1",
    "random_seed"       : SEED,
    "verbose"           : 0,
    "auto_class_weights": "Balanced"
})

print("Training CatBoost with best Optuna params + SMOTE...")
oof_cat, test_cat, cat_score = train_oof_cat(
    best_cat_params, X, y, X_test, sm=sm, label="CAT"
)

Training CatBoost with best Optuna params + SMOTE...
  [CAT] Fold 1 F1: 0.8009
  [CAT] Fold 2 F1: 0.8112
  [CAT] Fold 3 F1: 0.7801
  [CAT] Fold 4 F1: 0.7917
  [CAT] Fold 5 F1: 0.8096

  [CAT] OOF Macro F1: 0.7987
              precision    recall  f1-score   support

        High       0.78      0.66      0.72       470
         Low       0.93      0.88      0.90      6280
      Medium       0.72      0.83      0.77      2868

    accuracy                           0.85      9618
   macro avg       0.81      0.79      0.80      9618
weighted avg       0.86      0.85      0.86      9618



---
## 10. Ensemble + Threshold Optimization + Submission

### Ensemble Strategy
The three models are combined via **weighted average** where each model's weight is proportional to its OOF macro F1 score. This automatically gives more influence to stronger models without manual tuning.

### Threshold Optimization
Standard `argmax` on class probabilities doesn't optimize for macro F1 directly. Instead we learn per-class **probability multipliers** using Nelder-Mead optimization on the OOF predictions:

$$\hat{y} = \arg\max_k \frac{p_k}{t_k}$$

where $p_k$ is the predicted probability for class $k$ and $t_k$ is the learned threshold. Lowering a threshold makes the model more willing to predict that class — useful for boosting recall on the minority **High** class.

In [ ]:
def apply_thresholds(probs, thresholds):
    """Scale class probabilities by per-class thresholds then take argmax."""
    return np.argmax(probs / np.array(thresholds), axis=1)


def neg_macro_f1(thresholds, probs, y_true):
    """Negative macro F1 — minimized by scipy.optimize to find optimal thresholds."""
    return -macro_f1(y_true, apply_thresholds(probs, thresholds))

In [ ]:
# -------------------------------------------------------------------------
# Weighted ensemble — weights proportional to OOF F1 scores
# -------------------------------------------------------------------------
scores_arr = np.array([xgb_score, lgb_score, cat_score])
weights    = scores_arr / scores_arr.sum()

print(f"Ensemble weights — XGB: {weights[0]:.3f}, LGB: {weights[1]:.3f}, CAT: {weights[2]:.3f}")

oof_ensemble  = weights[0]*oof_xgb  + weights[1]*oof_lgb  + weights[2]*oof_cat
test_ensemble = weights[0]*test_xgb + weights[1]*test_lgb + weights[2]*test_cat

ensemble_score = macro_f1(y, np.argmax(oof_ensemble, axis=1))
print(f"\nEnsemble OOF F1 (default): {ensemble_score:.4f}")
print("\nPer-class breakdown (default):")
print(classification_report(y, np.argmax(oof_ensemble, axis=1), target_names=le_target.classes_))

# -------------------------------------------------------------------------
# Threshold optimization via Nelder-Mead
# Optimizes per-class multipliers directly on OOF macro F1
# -------------------------------------------------------------------------
print("Optimizing thresholds...")
result = minimize(
    neg_macro_f1,
    x0=[1.0, 1.0, 1.0],
    args=(oof_ensemble, y),
    method="Nelder-Mead",
    bounds=[(0.1, 2.0)] * 3,
    options={"maxiter": 5000, "xatol": 1e-5, "fatol": 1e-5}
)

best_thresholds = result.x
optimized_f1    = -result.fun

print(f"\nDefault   OOF F1 : {ensemble_score:.4f}")
print(f"Optimized OOF F1 : {optimized_f1:.4f}")
print(f"Thresholds — High: {best_thresholds[0]:.4f}, Low: {best_thresholds[1]:.4f}, Medium: {best_thresholds[2]:.4f}")

print("\nPer-class breakdown (optimized thresholds):")
print(classification_report(
    y, apply_thresholds(oof_ensemble, best_thresholds),
    target_names=le_target.classes_
))

# -------------------------------------------------------------------------
# Generate final submission
# -------------------------------------------------------------------------
final_preds = le_target.inverse_transform(apply_thresholds(test_ensemble, best_thresholds))
submission  = pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: final_preds})
submission.to_csv(OUTPUT_DIR / "submission_ensemble_clean.csv", index=False)

print(f"\nSaved: submission_ensemble_clean.csv")
print(f"Prediction distribution:\n{pd.Series(final_preds).value_counts()}")

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"  XGBoost   OOF F1 : {xgb_score:.4f}")
print(f"  LightGBM  OOF F1 : {lgb_score:.4f}")
print(f"  CatBoost  OOF F1 : {cat_score:.4f}")
print(f"  Ensemble  OOF F1 : {ensemble_score:.4f}")
print(f"  Optimized OOF F1 : {optimized_f1:.4f}")
print("=" * 50)

Ensemble weights — XGB: 0.334, LGB: 0.334, CAT: 0.332

Ensemble OOF F1 (default): 0.7996

Per-class breakdown (default):
              precision    recall  f1-score   support

        High       0.85      0.61      0.71       470
         Low       0.92      0.89      0.91      6280
      Medium       0.74      0.83      0.78      2868

    accuracy                           0.86      9618
   macro avg       0.84      0.78      0.80      9618
weighted avg       0.87      0.86      0.86      9618

Optimizing thresholds...

Default   OOF F1 : 0.7996
Optimized OOF F1 : 0.8082
Thresholds — High: 1.2534, Low: 0.6141, Medium: 1.1251

Per-class breakdown (optimized thresholds):
              precision    recall  f1-score   support

        High       0.88      0.60      0.71       470
         Low       0.88      0.98      0.93      6280
      Medium       0.88      0.70      0.78      2868

    accuracy                           0.88      9618
   macro avg       0.88      0.76      0.81     

---
## 11. Submission Preview

In [ ]:
submission

---
## Results

| Model | OOF Macro F1 |
|-------|-------------|
| XGBoost | 0.8050 |
| LightGBM | 0.8054 |
| CatBoost | 0.7987 |
| **Ensemble (default)** | 0.7996 |
| **Ensemble (optimized thresholds)** | **0.8082** |

| Split | Score |
|-------|-------|
| Public Leaderboard | 0.8847 |
| **Private Leaderboard** | **0.8860** ✅ |

---

## Key Takeaways

1. **Data cleaning was the most impactful change** — the apostrophe normalization fix alone unlocked signal in 9 status columns that were silently producing `NaN` before any modeling.

2. **Missing value flags are predictive** — `missing_business_age_years` was consistently the 2nd most important SHAP feature, because Lesotho respondents' missingness pattern correlates strongly with Low FHI.

3. **Simpler pipelines generalize better** — more complex experiments (pseudo-labeling, target encoding, additional features) improved the public leaderboard score but this simpler 3-model ensemble achieved the best private leaderboard score, suggesting the added complexity caused mild overfitting to the public test set.

4. **Threshold optimization is free** — moving from `argmax` to per-class threshold optimization added +0.0086 macro F1 with zero retraining cost.

5. **Hardware matters for XGBoost** — GPU and CPU produce different results due to floating-point parallelization differences in XGBoost's `hist` method. This notebook was run on a Kaggle T4 GPU.